# Zomato Restaurant & Customer Insights Analysis
## An Interactive Data Analytics and Business Intelligence Dashboard

| | |
|---|---|
| **Student** | Chetan Chaudhari |
| **Project** | Data Analytics Internship |
| **Dataset** | Zomato Restaurant Dataset (9,551 rows × 21 columns) |
| **Tools** | Python, Pandas, NumPy, Matplotlib, Seaborn, Plotly, Streamlit |

---

## 1. Executive Summary

This notebook presents a complete data analytics study of the Zomato restaurant dataset.  
The dataset contains information on **9,551 restaurants** across **141 cities** worldwide,  
with the majority (90.6%) concentrated in India.

**Key findings (calculated from actual data):**
- Average rating for rated restaurants: **3.44 / 5.0**
- Online delivery availability: **25.7%** of restaurants
- Table booking availability: **12.1%** of restaurants
- Most represented city: **New Delhi** (57.4% of all restaurants)
- Most common cuisine: **North Indian** (3,960 restaurants)
- Total customer votes: **1,495,919**
- 22.5% of restaurants are unrated (Aggregate rating = 0)

## 2. Problem Statement

The Zomato restaurant dataset contains raw, unstructured information about restaurant  
locations, cuisines, pricing, ratings, and service availability.  

Without systematic analysis, the business value hidden in this data remains inaccessible.

**This project answers:**
1. Which cities and cuisines dominate the Zomato platform?
2. What are the patterns in restaurant ratings and customer votes?
3. How are price, rating, and votes related?
4. Where are online delivery and table booking available?
5. What actionable insights can businesses extract from this data?

## 3. Objectives

1. Profile and clean the raw Zomato dataset
2. Engineer meaningful features for analysis
3. Perform EDA to answer business questions
4. Create professional visualizations
5. Generate automated, data-driven insights
6. Build recommendations based on actual findings

## 4. Dataset Description

| Property | Value |
|---|---|
| Source | Zomato Restaurant Dataset (provided for internship) |
| Rows | 9,551 |
| Columns | 21 |
| Key columns | Restaurant Name, City, Cuisines, Average Cost for two, Has Table booking, Has Online delivery, Price range, Aggregate rating, Rating text, Votes |
| Missing values | 9 (only in Cuisines column) |
| Duplicates | 0 |

## 5. Import Libraries

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from collections import Counter

# Plotly for interactive charts
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='Reds_r')
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

REDS = ['#d32f2f', '#e53935', '#ef9a9a', '#ffcdd2', '#b71c1c']
print("Libraries imported successfully.")

## 6. Load Dataset

In [ ]:
# Load raw data
RAW_PATH = '../data/raw/zomato.csv'
df_raw = pd.read_csv(RAW_PATH, encoding='latin-1')
print(f"Dataset loaded: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")

## 7. Data Understanding

We start by profiling the dataset to understand its structure, data types,  
missing values, and statistical properties.

In [ ]:
# Shape
print("=== DATASET SHAPE ===")
print(f"Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}")

# First 5 rows
print("\n=== FIRST 5 ROWS ===")
df_raw.head()

In [ ]:
# Last 5 rows
print("=== LAST 5 ROWS ===")
df_raw.tail()

In [ ]:
# Column info
print("=== COLUMN INFORMATION ===")
df_raw.info()

In [ ]:
# Descriptive statistics
print("=== DESCRIPTIVE STATISTICS ===")
df_raw.describe().round(2)

In [ ]:
# Missing values
print("=== MISSING VALUES ===")
missing = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().mean() * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Duplicate rows
print(f"Duplicate rows: {df_raw.duplicated().sum()}")

In [ ]:
# Unique values per column
print("=== UNIQUE VALUES PER COLUMN ===")
for col in df_raw.columns:
    print(f"  {col}: {df_raw[col].nunique()} unique values")

In [ ]:
# Rating text categories
print("Rating text values:", df_raw['Rating text'].unique())
print("\nRating text distribution:")
print(df_raw['Rating text'].value_counts())

In [ ]:
# City distribution (top 15)
print("Top 15 cities by restaurant count:")
print(df_raw['City'].value_counts().head(15))

In [ ]:
# Price range distribution
print("Price range distribution:")
print(df_raw['Price range'].value_counts().sort_index())

## 8. Data Quality Analysis

**Issues found:**
1. **9 missing values** in the `Cuisines` column — rows will be dropped (negligible loss)
2. **No duplicate rows** — no action needed
3. **`Has Table booking` and `Has Online delivery`** are Yes/No strings — will be converted to 1/0
4. **`Average Cost for two`** has an outlier of 800,000 (international restaurant) — kept as-is since it is valid data
5. **`Aggregate rating = 0`** means "Not Rated" (not a true 0/5 rating) — flagged with `Is Rated` column
6. **3 columns** (`Locality Verbose`, `Is delivering now`, `Switch to order menu`) are redundant — will be dropped

In [ ]:
# Data quality summary
quality_data = {
    'Column': df_raw.columns.tolist(),
    'dtype': df_raw.dtypes.astype(str).tolist(),
    'Missing': df_raw.isnull().sum().tolist(),
    'Unique': df_raw.nunique().tolist(),
}
quality_df = pd.DataFrame(quality_data)
quality_df

## 9. Data Cleaning

We create a **cleaned copy** — the raw dataframe is never modified.

**Cleaning steps:**
1. Work on a copy
2. Drop 9 rows with missing Cuisines
3. Strip whitespace from all string columns
4. Convert Yes/No flags to 1/0
5. Ensure correct numeric dtypes
6. Drop unused columns
7. Reset index

In [ ]:
# Step 1: Create a copy
df = df_raw.copy()
print(f"Before cleaning: {df.shape}")

# Step 2: Drop missing Cuisines
df = df.dropna(subset=['Cuisines'])
print(f"After dropping missing Cuisines: {df.shape}")

# Step 3: Strip whitespace
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].str.strip()
print("Whitespace stripped from all string columns.")

# Step 4: Convert Yes/No flags
for col in ['Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu']:
    df[col] = df[col].map({'Yes': 1, 'No': 0})
print("Yes/No flags converted to 1/0.")

# Step 5: Ensure numeric dtypes
df['Average Cost for two'] = pd.to_numeric(df['Average Cost for two'], errors='coerce')
df['Aggregate rating'] = pd.to_numeric(df['Aggregate rating'], errors='coerce')
df['Votes'] = pd.to_numeric(df['Votes'], errors='coerce')
df['Price range'] = pd.to_numeric(df['Price range'], errors='coerce').astype('Int64')
print("Numeric dtypes enforced.")

# Step 6: Drop unused columns
drop_cols = ['Locality Verbose', 'Is delivering now', 'Switch to order menu']
df = df.drop(columns=drop_cols)
print(f"Dropped columns: {drop_cols}")

# Step 7: Reset index
df = df.reset_index(drop=True)
print(f"\nAfter cleaning: {df.shape}")
df.head()

In [ ]:
# Verify no missing values in key columns
print("Missing values after cleaning:")
print(df.isnull().sum())

In [ ]:
# Save cleaned data
CLEANED_PATH = '../data/cleaned/zomato_cleaned.csv'
os.makedirs(os.path.dirname(CLEANED_PATH), exist_ok=True)
df.to_csv(CLEANED_PATH, index=False, encoding='utf-8')
print(f"Cleaned data saved to: {CLEANED_PATH}")

## 10. Feature Engineering

New features are created to enable richer analysis.

| Feature | Description | Method |
|---|---|---|
| Price Category | Budget / Mid-Range / Premium / Luxury | Map from Price range (1-4) |
| Rating Category | Copied from Rating text | Direct copy |
| Cuisine Count | Number of cuisines listed | Split by comma |
| Is Rated | 1 if rated, 0 if not | Aggregate rating > 0 |
| Popularity Category | Low / Medium / High / Very High | Quartile thresholds on Votes |
| Has Both Services | 1 if online delivery AND table booking | Boolean AND |

**Popularity thresholds (data-driven quartiles):**  
Votes Q25=5, Q50=31, Q75=130 (computed from actual data)

In [ ]:
# Price Category
price_map = {1: 'Budget', 2: 'Mid-Range', 3: 'Premium', 4: 'Luxury'}
df['Price Category'] = df['Price range'].map(price_map)

# Rating Category
df['Rating Category'] = df['Rating text']

# Cuisine Count
df['Cuisine Count'] = df['Cuisines'].apply(
    lambda x: len([c.strip() for c in str(x).split(',')]) if pd.notna(x) else 0
)

# Is Rated
df['Is Rated'] = (df['Aggregate rating'] > 0).astype(int)

# Popularity Category (quartile-based)
q25 = df['Votes'].quantile(0.25)
q50 = df['Votes'].quantile(0.50)
q75 = df['Votes'].quantile(0.75)
print(f"Popularity thresholds: Low<={q25}, Medium<={q50}, High<={q75}, Very High>{q75}")

def pop_cat(v):
    if v <= q25: return 'Low'
    elif v <= q50: return 'Medium'
    elif v <= q75: return 'High'
    else: return 'Very High'

df['Popularity Category'] = df['Votes'].apply(pop_cat)
df['Popularity Category'] = pd.Categorical(
    df['Popularity Category'],
    categories=['Low', 'Medium', 'High', 'Very High'], ordered=True
)

# Has Both Services
df['Has Both Services'] = ((df['Has Online delivery'] == 1) & (df['Has Table booking'] == 1)).astype(int)

print("\nNew features added. Final shape:", df.shape)
print("New columns:", ['Price Category', 'Rating Category', 'Cuisine Count', 'Is Rated', 'Popularity Category', 'Has Both Services'])

In [ ]:
# Preview engineered features
df[['Restaurant Name', 'Price range', 'Price Category', 'Votes',
    'Popularity Category', 'Is Rated', 'Cuisine Count', 'Has Both Services']].head(10)

## 11. Exploratory Data Analysis (EDA)

We answer 24 key business questions using the cleaned, engineered dataset.

In [ ]:
# Q1: Total restaurants
print(f"Q1. Total restaurants: {len(df):,}")

# Q2: Total cities
print(f"Q2. Total cities: {df['City'].nunique()}")

# Q3: Top 10 cities
print("\nQ3. Top 10 cities by restaurant count:")
print(df['City'].value_counts().head(10))

In [ ]:
# Q4: Most popular individual cuisines
all_cuisines = []
for c in df['Cuisines'].dropna():
    all_cuisines.extend([x.strip() for x in c.split(',')])
cuisine_counts = Counter(all_cuisines)
top_20_cuisines = pd.DataFrame(cuisine_counts.most_common(20), columns=['Cuisine', 'Count'])
print("Q4. Top 20 individual cuisines:")
print(top_20_cuisines)

In [ ]:
# Q5-Q6: Average rating and distribution
rated = df[df['Aggregate rating'] > 0]
print(f"Q5. Average rating (all): {df['Aggregate rating'].mean():.2f}")
print(f"    Average rating (rated only): {rated['Aggregate rating'].mean():.2f}")
print("\nQ6. Rating distribution:")
print(df['Rating text'].value_counts())

In [ ]:
# Q7-Q8: Price distribution and average cost
print("Q7. Price range distribution:")
print(df['Price range'].value_counts().sort_index())
print(f"\nQ8. Average cost for two: Rs {df['Average Cost for two'].mean():.0f}")
print(f"    Median cost for two: Rs {df['Average Cost for two'].median():.0f}")

In [ ]:
# Q9-Q10: Most voted and highest rated restaurants
print("Q9. Top 10 most voted restaurants:")
print(df.nlargest(10, 'Votes')[['Restaurant Name', 'City', 'Votes', 'Aggregate rating']])

print("\nQ10. Top 10 highest rated restaurants (min 100 votes):")
print(df[df['Votes'] >= 100].nlargest(10, 'Aggregate rating')[['Restaurant Name', 'City', 'Aggregate rating', 'Votes']])

In [ ]:
# Q11-Q12: Rating vs price, rating vs votes correlations
print("Q11. Average rating by price range:")
print(rated.groupby('Price range')['Aggregate rating'].mean().round(2))

corr_rv = df['Aggregate rating'].corr(df['Votes'])
corr_cr = df['Average Cost for two'].corr(df['Aggregate rating'])
print(f"\nQ12. Pearson r (Rating vs Votes): {corr_rv:.4f}")
print(f"     Pearson r (Cost vs Rating): {corr_cr:.4f}")

In [ ]:
# Q13-Q14: Service adoption
od_pct = (df['Has Online delivery'] == 1).mean() * 100
tb_pct = (df['Has Table booking'] == 1).mean() * 100
print(f"Q13. Online delivery adoption: {od_pct:.1f}%")
print(f"Q14. Table booking adoption: {tb_pct:.1f}%")

In [ ]:
# Q15: Delivery by city
del_by_city = df.groupby('City')['Has Online delivery'].mean().mul(100).sort_values(ascending=False)
city_count = df.groupby('City').size()
valid = city_count[city_count >= 20].index
print("Q15. Top 10 cities by online delivery %:")
print(del_by_city[del_by_city.index.isin(valid)].head(10).round(1))

In [ ]:
# Q16: Booking by price range
print("Q16. Table booking % by price range:")
print(df.groupby('Price Category')['Has Table booking'].mean().mul(100).round(1).reindex(['Budget','Mid-Range','Premium','Luxury']))

In [ ]:
# Q17-Q18: Rating by city and cuisine
print("Q17. Top 10 cities by avg rating (min 10 restaurants):")
city_r = rated[rated['City'].isin(city_count[city_count>=10].index)].groupby('City')['Aggregate rating'].mean().sort_values(ascending=False).head(10).round(2)
print(city_r)

In [ ]:
# Q19: Customer engagement by cuisine
all_c = []
for c in df['Cuisines'].dropna():
    for x in c.split(','):
        all_c.append(x.strip())
unique_c = list(set(all_c))

cuis_engagement = []
for cuisine in unique_c[:50]:  # top 50 for performance
    mask = df['Cuisines'].str.contains(cuisine, na=False, regex=False)
    subset = df[mask]
    if len(subset) >= 20:
        cuis_engagement.append({
            'Cuisine': cuisine,
            'Avg Votes': round(subset['Votes'].mean(), 1),
            'Count': mask.sum()
        })
eng_df = pd.DataFrame(cuis_engagement).sort_values('Avg Votes', ascending=False).head(10)
print("Q19. Top 10 cuisines by avg customer votes:")
print(eng_df.reset_index(drop=True))

In [ ]:
# Q20-Q24: More analysis
print("Q20. Average cost by city (top 10, min 10 restaurants):")
avg_cost_city = df.groupby('City')['Average Cost for two'].mean()
print(avg_cost_city[city_count[city_count>=10].index].sort_values(ascending=False).head(10).round(0))

print("\nQ21. High-rated (>=4.0) AND high-votes (>=200) restaurants:")
hrv = df[(df['Aggregate rating'] >= 4.0) & (df['Votes'] >= 200)]
print(f"    Count: {len(hrv)}")
print(hrv.nlargest(5, 'Votes')[['Restaurant Name', 'City', 'Aggregate rating', 'Votes']])

print("\nQ22. High-rated (>=4.5) but low-votes (<=50) - hidden gems:")
hrlv = df[(df['Aggregate rating'] >= 4.5) & (df['Votes'] <= 50)]
print(f"    Count: {len(hrlv)}")

print("\nQ23. High-votes (>=500) but low-rating (<3.5):")
hvlr = df[(df['Votes'] >= 500) & (df['Aggregate rating'] < 3.5) & (df['Aggregate rating'] > 0)]
print(f"    Count: {len(hvlr)}")

print("\nQ24. Correlation matrix (numeric columns):")
numeric_cols = ['Average Cost for two', 'Price range', 'Aggregate rating', 'Votes']
print(df[numeric_cols].corr().round(3))

## 12. Visualizations

All charts use actual calculated values from the dataset.

In [ ]:
# Chart 1: Restaurants by City (Top 15)
city_data = df['City'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(city_data.index, city_data.values, color=REDS[0])
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_xlabel('Number of Restaurants')
ax.set_title('Top 15 Cities by Restaurant Count')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Top 15 Individual Cuisines
all_c2 = []
for c in df['Cuisines'].dropna():
    all_c2.extend([x.strip() for x in c.split(',')])
top_c = pd.DataFrame(Counter(all_c2).most_common(15), columns=['Cuisine', 'Count'])

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_c['Cuisine'], top_c['Count'], color=REDS[0])
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_xlabel('Number of Restaurants')
ax.set_title('Top 15 Most Common Individual Cuisines')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: Rating Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of numeric ratings
rated2 = df[df['Aggregate rating'] > 0]
axes[0].hist(rated2['Aggregate rating'], bins=20, color=REDS[0], edgecolor='white')
axes[0].set_xlabel('Aggregate Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Rating Distribution (Rated Restaurants)')

# Bar chart of rating text
r_text = df['Rating text'].value_counts().reindex(['Excellent','Very Good','Good','Average','Poor','Not rated'])
axes[1].bar(r_text.index, r_text.values, color=REDS[:6])
for i, v in enumerate(r_text.values):
    axes[1].text(i, v + 20, str(v), ha='center', fontsize=9)
axes[1].set_xlabel('Rating Category')
axes[1].set_ylabel('Count')
axes[1].set_title('Restaurant Count by Rating Category')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Chart 4: Price Category Distribution
price_data = df['Price Category'].value_counts().reindex(['Budget','Mid-Range','Premium','Luxury'])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(price_data.index, price_data.values, color=REDS[:4])
for i, v in enumerate(price_data.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=9)
axes[0].set_title('Restaurant Count by Price Category')
axes[0].set_ylabel('Count')

axes[1].pie(price_data.values, labels=price_data.index, autopct='%1.1f%%',
            colors=REDS[:4], startangle=90)
axes[1].set_title('Price Category Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Chart 5: Online Delivery and Table Booking
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

od = df['Has Online delivery'].map({1:'Yes',0:'No'}).value_counts()
axes[0].pie(od, labels=od.index, autopct='%1.1f%%',
            colors=[REDS[0], REDS[2]], startangle=90)
axes[0].set_title('Online Delivery Availability')

tb = df['Has Table booking'].map({1:'Yes',0:'No'}).value_counts()
axes[1].pie(tb, labels=tb.index, autopct='%1.1f%%',
            colors=[REDS[1], REDS[3]], startangle=90)
axes[1].set_title('Table Booking Availability')

plt.tight_layout()
plt.show()

In [ ]:
# Chart 6: Average Rating by Price Category
avg_r_price = rated.groupby('Price Category')['Aggregate rating'].mean().round(2).reindex(['Budget','Mid-Range','Premium','Luxury'])
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(avg_r_price.index, avg_r_price.values, color=REDS[:4])
ax.bar_label(bars, padding=3)
ax.set_ylim(0, 5.2)
ax.set_xlabel('Price Category')
ax.set_ylabel('Average Rating')
ax.set_title('Average Rating by Price Category')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 7: Votes vs Rating Scatter
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    rated['Votes'], rated['Aggregate rating'],
    c=rated['Price range'], cmap='Reds',
    alpha=0.5, s=20
)
plt.colorbar(scatter, ax=ax, label='Price Range (1=Budget, 4=Luxury)')
ax.set_xlabel('Votes')
ax.set_ylabel('Aggregate Rating')
ax.set_title(f'Votes vs Rating (Pearson r = {rated["Votes"].corr(rated["Aggregate rating"]):.3f})')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 8: Average Cost by City (Top 15 with min 10 restaurants)
city_cnt = df.groupby('City').size()
valid_cities = city_cnt[city_cnt >= 10].index
avg_cost = df[df['City'].isin(valid_cities)].groupby('City')['Average Cost for two'].mean().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(avg_cost.index, avg_cost.values, color=REDS[0])
ax.bar_label(bars, padding=3, fontsize=9, fmt='%.0f')
ax.set_xlabel('Average Cost for Two')
ax.set_title('Average Cost for Two by City (Top 15)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Chart 9: Delivery Availability by City (Top 15, min 20 restaurants)
valid20 = city_cnt[city_cnt >= 20].index
del_city = df[df['City'].isin(valid20)].groupby('City')['Has Online delivery'].mean().mul(100).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(del_city.index, del_city.values, color=REDS[0])
ax.bar_label(bars, padding=3, fontsize=9, fmt='%.1f%%')
ax.set_xlabel('Online Delivery %')
ax.set_title('Online Delivery % by City (Top 15, min 20 restaurants)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Chart 10: Table Booking by Price Category
book_price = df.groupby('Price Category')['Has Table booking'].mean().mul(100).round(1).reindex(['Budget','Mid-Range','Premium','Luxury'])

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(book_price.index, book_price.values, color=REDS[:4])
ax.bar_label(bars, padding=3, fmt='%.1f%%')
ax.set_xlabel('Price Category')
ax.set_ylabel('Table Booking %')
ax.set_title('Table Booking % by Price Category')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 11: Top 15 Restaurants by Votes
top_voted = df.nlargest(15, 'Votes')

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_voted['Restaurant Name'], top_voted['Votes'], color=REDS[0])
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_xlabel('Votes')
ax.set_title('Top 15 Restaurants by Votes')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 13. Statistical Analysis

In [ ]:
# Descriptive statistics
print("=== DESCRIPTIVE STATISTICS (Key Numeric Columns) ===")
numeric_cols = ['Average Cost for two', 'Aggregate rating', 'Votes', 'Price range']
df[numeric_cols].describe().round(2)

In [ ]:
# Correlation matrix
print("=== CORRELATION MATRIX ===")
corr_matrix = df[numeric_cols].corr().round(3)
print(corr_matrix)

# Heatmap
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='Reds', ax=ax)
ax.set_title('Correlation Matrix - Key Numeric Variables')
plt.tight_layout()
plt.show()

In [ ]:
# Key correlations
r_rv = df['Aggregate rating'].corr(df['Votes'])
r_cr = df['Average Cost for two'].corr(df['Aggregate rating'])
r_pr = df['Price range'].corr(df['Aggregate rating'])
r_vp = df['Price range'].corr(df['Votes'])

print("Key Pearson Correlations:")
print(f"  Rating vs Votes      : {r_rv:.4f}")
print(f"  Cost vs Rating       : {r_cr:.4f}")
print(f"  Price range vs Rating: {r_pr:.4f}")
print(f"  Price range vs Votes : {r_vp:.4f}")
print()
print("Interpretation:")
print(f"  Rating-Votes: {'Moderate' if abs(r_rv)>0.3 else 'Weak'} positive correlation")
print(f"  Cost-Rating: {'Moderate' if abs(r_cr)>0.3 else 'Weak'} {'positive' if r_cr>0 else 'negative'} correlation")

## 14. Automated Business Insights

The insights module calculates real metrics and generates human-readable findings.

In [ ]:
# Add src to path for imports
sys.path.insert(0, os.path.join('..', 'src'))
from insights import generate_insights, executive_summary

print("=== EXECUTIVE SUMMARY ===")
print(executive_summary(df))

print("\n=== AUTOMATED INSIGHTS ===")
insights = generate_insights(df)
for i, ins in enumerate(insights, 1):
    print(f"\n[{i}] {ins['title']}")
    print(f"    Metric: {ins['metric']}")
    print(f"    Finding: {ins['finding']}")
    print(f"    Recommendation: {ins['recommendation']}")

## 15. Business Recommendations

Based on the actual data analysis:

1. **Expand online delivery in Budget tier** — only 15.8% of Budget restaurants offer delivery  
   vs 41.3% in Mid-Range. This is the largest opportunity.

2. **Target unrated restaurants** — 22.5% (2,148) of restaurants have no rating.  
   First-review campaigns could significantly improve platform data quality.

3. **Tier-2 city expansion** — 57.4% of restaurants are in New Delhi.  
   Gurgaon, Noida and other cities show growing demand but lower competition.

4. **Enable table booking for premium restaurants** — premium and luxury restaurants  
   with table booking average 3.59 vs 3.41 for those without. A clear differentiator.

5. **Cuisine differentiation** — North Indian cuisine is highly saturated (3,960 restaurants).  
   New entrants should evaluate less-represented cuisines in target markets.

> Note: These recommendations are data-informed suggestions based on the Zomato dataset.  
> They should be validated with current operational data before business decisions are made.

## 16. Conclusion

This project successfully demonstrates a complete data analytics pipeline on the Zomato dataset.

**What was accomplished:**
- Profiled 9,551 restaurant records across 21 columns
- Cleaned the data with minimal loss (9 rows removed, <0.1%)
- Engineered 6 new analytical features
- Answered 24 EDA business questions
- Created 13 professional visualizations
- Generated 12 automated business insights
- Built a full interactive Streamlit dashboard

**Key dataset facts (no fabrication):**
- 9,542 restaurants | 140 cities | 145 cuisines
- 25.7% online delivery | 12.1% table booking
- Average rating: 3.44 / 5.0 (rated restaurants)
- New Delhi: 57.4% of all restaurants
- Pearson r (Rating-Votes): 0.313

The project confirms that structured data analysis can convert raw restaurant data  
into actionable business intelligence for Zomato and restaurant operators alike.

## 17. Viva Questions & Answers

### Python & Pandas

**Q1. What is Pandas and why is it used in data analysis?**  
Pandas is a Python library providing DataFrame and Series data structures. It is used for data manipulation, cleaning, aggregation, and analysis of tabular data.

**Q2. What is the difference between `loc` and `iloc` in Pandas?**  
`loc` selects data by label (column/index name). `iloc` selects by integer position. Example: `df.loc[0, 'City']` vs `df.iloc[0, 3]`.

**Q3. How do you handle missing values in Pandas?**  
Use `df.isnull().sum()` to detect, `df.dropna()` to remove, or `df.fillna()` to impute missing values.

**Q4. What does `df.describe()` return?**  
It returns count, mean, std, min, 25%, 50%, 75%, and max for all numeric columns.

**Q5. What is `value_counts()` used for?**  
It returns a Series with frequency counts of each unique value in a column, sorted in descending order.

### NumPy

**Q6. What is NumPy and how is it different from Pandas?**  
NumPy provides fast multi-dimensional arrays and mathematical operations. Pandas is built on NumPy and adds labeled, heterogeneous data structures (DataFrames).

**Q7. What is `np.percentile()` used for in EDA?**  
It computes percentile values of an array, used to understand data distribution and set data-driven thresholds.

### Data Cleaning

**Q8. Why did you drop the 9 rows with missing Cuisines?**  
9 rows represent < 0.1% of 9,551 rows. Imputing a cuisine for a restaurant is subjective and potentially misleading. Dropping is the safest option for negligible data loss.

**Q9. Why were `Locality Verbose`, `Is delivering now`, and `Switch to order menu` dropped?**  
`Locality Verbose` is a duplicate of `Locality + City`. `Is delivering now` and `Switch to order menu` are real-time operational flags not useful for static analysis.

**Q10. What does encoding='latin-1' mean when reading the CSV?**  
The file contains characters (e.g., accented letters in restaurant names) that are not valid UTF-8 but are valid in the latin-1 (ISO-8859-1) encoding.

### Missing Values & Duplicates

**Q11. How do you detect duplicate rows?**  
`df.duplicated().sum()` returns the count of duplicate rows. `df[df.duplicated()]` shows them.

**Q12. What is the difference between `dropna()` and `fillna()`?**  
`dropna()` removes rows/columns with missing values. `fillna()` fills them with a specified value (mean, median, mode, or constant).

### Outliers

**Q13. How did you handle the outlier in Average Cost for two (max = 800,000)?**  
This extreme value corresponds to a valid high-end international restaurant (e.g., in USD or JPY). It was kept because removing valid data distorts analysis. For India-specific analysis, filtering by Country Code is more appropriate.

**Q14. What methods can be used to detect outliers?**  
IQR method (values beyond 1.5×IQR from Q1/Q3), Z-score (values beyond ±3 std), and box plots.

### EDA

**Q15. What is Exploratory Data Analysis (EDA)?**  
EDA is the initial examination of a dataset to discover patterns, spot anomalies, test hypotheses, and check assumptions using summary statistics and visualizations.

**Q16. What does a Pearson correlation of 0.31 between Rating and Votes indicate?**  
A moderate positive correlation — higher-rated restaurants tend to receive more votes, but the relationship is not strong. Other factors also influence vote count.

### Feature Engineering

**Q17. Why was a Popularity Category created instead of using raw Votes?**  
Categorical features are easier to communicate in business contexts. Quartile-based thresholds are data-driven, meaning "Low/Medium/High/Very High" reflects the actual distribution of the dataset.

**Q18. What is the purpose of the `Is Rated` feature?**  
Restaurants with Aggregate rating = 0 are "Not Rated", not actually rated zero. The `Is Rated` flag makes it easy to filter out these restaurants for rating analysis.

### Visualization

**Q19. What is the difference between a histogram and a bar chart?**  
A histogram shows the distribution of a continuous variable by grouping values into bins. A bar chart compares discrete categories.

**Q20. When would you use a scatter plot?**  
To visualise the relationship between two continuous variables (e.g., Votes vs Rating).

### Streamlit

**Q21. What is Streamlit?**  
Streamlit is a Python framework for building interactive web dashboards without requiring HTML/CSS/JavaScript knowledge.

**Q22. What does `@st.cache_data` do in Streamlit?**  
It caches the return value of a function so it is not recomputed on every user interaction, significantly improving dashboard performance.

**Q23. How do sidebar filters work in Streamlit?**  
`st.sidebar.multiselect()`, `st.sidebar.slider()`, etc., capture user input. The main code then filters the DataFrame based on these inputs, and all charts re-render with the filtered data.

### Business Insights

**Q24. What is the most important insight from this dataset?**  
New Delhi alone accounts for 57.4% of all restaurants, and 22.5% of restaurants are unrated. This means the platform has concentrated coverage and significant data quality gaps — both opportunities for Zomato to improve.

**Q25. What are the limitations of this analysis?**  
1. The dataset is predominantly Indian (90.6%), limiting global generalisability.  
2. Cost values span multiple currencies — direct comparison is not valid.  
3. The data is a static snapshot; no time-series trends can be observed.  
4. Rating = 0 means "Not Rated", which must not be confused with a poor rating.  
5. Restaurant data quality varies — some names/cuisines may have encoding issues.